# Recover SCST1500 VALID rerank on CPU

Choose Accelerator None, Internet and enable Kaggle Secret codex. The source is pinned to ducanh3/notebook7bc83c4c76/94. Only its review ZIP is downloaded. Saved beams and CLIP scores are replayed with the original scoring rule.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/94'
SECRET_NAME='codex'
import re
if not re.fullmatch(r'[^/]+/[^/]+/[1-9][0-9]*',SOURCE_VERSION):
    raise ValueError('Set SOURCE_VERSION to the exact SCST1500 Save Version')
OUTPUT=Path('/kaggle/working/csa_scst_only1500_rerank_cpu_recovery')
OUTPUT.mkdir()
SOURCE=Path(tempfile.mkdtemp(prefix='scst1500-cpu-',dir='/tmp'))
DEPS=SOURCE/'deps'; CLI=SOURCE/'cli'; CHAIR=SOURCE/'chair'; LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+7200
TOKENS=[]
os.environ.update(PYTHONPATH=str(SOURCE),OMP_NUM_THREADS='1')
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    from kaggle_secrets import UserSecretsClient
    try:
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    except Exception:
        raise RuntimeError('Enable the named Kaggle API Secret with access to the source notebook') from None
    if not token:
        raise RuntimeError('The named Kaggle API Secret is empty')
    TOKENS.append(token)
    cli_env=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
        cli_env.pop(key,None)
    run([sys.executable,'-m','pip','install','-q','--target',CLI,'kaggle==2.2.5'],300,cli_env)
    cli_env.update(KAGGLE_API_TOKEN=token,PYTHONPATH=str(CLI)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1')
    downloaded=SOURCE/'download'
    downloaded.mkdir()
    run([sys.executable,'-u','-c','from kaggle.cli import main; main()',
         'kernels','output',SOURCE_VERSION,'-p',downloaded,
         '--file-pattern',r'(^|/)csa_scst_only1500_rerank_valid5000_review[.]zip$'],900,cli_env)
    archive,=downloaded.rglob('csa_scst_only1500_rerank_valid5000_review.zip')
    rerank_out=OUTPUT/'rerank'
    rerank_out.mkdir()
    required=['pilot_checkpoint_receipt.json','checkpoint_source.json','vocabulary.json','test_lock.json',
              'rerank/test_lock.json','rerank/inference_inputs.json','rerank/test_references.json']
    for rank in (0,1):
        required.extend([f'worker{rank}.json',f'beams_rank{rank}.json',f'captions_rank{rank}.json',
                         f'rerank/worker{rank}.json',f'rerank/scores_rank{rank}.jsonl'])
    with zipfile.ZipFile(archive) as bundle:
        for name in required:
            with bundle.open(name) as source,(OUTPUT/name).open('wb') as target:
                shutil.copyfileobj(source,target)
    receipt=json.loads((OUTPUT/'pilot_checkpoint_receipt.json').read_text())
    source=json.loads((OUTPUT/'checkpoint_source.json').read_text())
    CHECKPOINT_SHA='b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef'
    if (receipt['updates']!=1500 or receipt['parent_sha256']!=source['sha256']
            or source['sha256']!='56a90b2223b3bb64e433a342d048389dc3e80693fa574c2c1543e31e39a4377b'
            or receipt['source_sha256']!=CHECKPOINT_SHA):
        raise ValueError('Wrong saved SCST1500 checkpoint lineage')
    (OUTPUT/'recovery_source.json').write_text(json.dumps(dict(kaggle_source=SOURCE_VERSION,receipt=receipt)))
    for path,name in MAPPING.items():
        if not name.endswith(('.py','.json')):
            continue
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],300)
    for path,digest in {'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/277851d0fbc4643ae7001bc7702dd0ce5c91d0f6/'+path,digest,target)
    (SOURCE/'REPLAY_RECEIPTS.json').write_text(json.dumps(dict(checkpoint_sha256=receipt['sha256'], xe_source_sha256=CHECKPOINT_SHA)))
    (SOURCE/'csa_per_caption_rerank_data.py').write_text('"""Prepare and check inputs for the frozen CSA-only rerank replay."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom captioning.object_consistency import index_detections\nfrom patch_beam import text_from_ids\nfrom real_bank_data import read, save, sha\nfrom test199_data import DATA_SHA\n\nHERE = Path(__file__).resolve().parent\nPOLICIES = (\'decoder_rank0\', \'clip_object_fixed\')\nDETECTION_SHA = \'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c\'\n\n\ndef prepare(cfg):\n    out = Path(cfg[\'output\'])\n    test = Path(cfg[\'test_output\'])\n    source = read(test/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    receipt = read(test/\'pilot_checkpoint_receipt.json\')\n    if (source[\'checkpoint_sha256\'] != replay[\'xe_source_sha256\']\n            or receipt[\'source_sha256\'] != replay[\'xe_source_sha256\']\n            or receipt[\'sha256\'] != replay[\'checkpoint_sha256\'] or receipt[\'updates\'] != 1500):\n        raise ValueError(\'Changed CSA checkpoint\')\n    if sha(cfg[\'dataset_json\']) != DATA_SHA or sha(cfg[\'detections\']) != DETECTION_SHA:\n        raise ValueError(\'Changed COCO metadata or detector\')\n    records = sorted((r for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'val\'),\n                     key=lambda r: int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in records] != source[\'expected_test_ids\']:\n        raise ValueError(\'Wrong TEST image cohort\')\n    detections = index_detections(read(cfg[\'detections\']))\n    root = Path(cfg[\'dataset_json\']).parent.resolve()\n    names = {r[\'filename\'] for r in records}\n    paths = {}\n    for p in root.rglob(\'*.jpg\'):\n        if p.name in names:\n            if p.name in paths or not p.resolve().is_relative_to(root):\n                raise ValueError(\'Ambiguous TEST JPEG: \'+p.name)\n            paths[p.name] = p\n    if set(paths) != names or not names <= detections.keys():\n        raise ValueError(\'Missing TEST JPEGs or detector rows\')\n    inputs = []\n    for row in records:\n        name = row[\'filename\']\n        path = paths[name]\n        inputs.append(dict(coco_id=int(row[\'cocoid\']), image_path=str(path),\n                           image_sha256=sha(path), detected=detections[name]))\n    save(out/\'inference_inputs.json\', inputs)\n    (out/\'test_references.json\').write_bytes((test/\'test_references.json\').read_bytes())\n    save(out/\'test_lock.json\', dict(\n        checkpoint_sha256=replay[\'checkpoint_sha256\'], images=5000,\n        clip=policy[\'clip\'], selector=policy[\'frozen_selector\'],\n        inference_sha256=sha(out/\'inference_inputs.json\'),\n        references_sha256=sha(out/\'test_references.json\')))\n\n\ndef checked_lock(out):\n    out = Path(out)\n    lock = read(out/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if (lock[\'clip\'], lock[\'selector\'], lock[\'checkpoint_sha256\'], lock[\'images\']) != (\n            policy[\'clip\'], policy[\'frozen_selector\'], replay[\'checkpoint_sha256\'], 5000):\n        raise ValueError(\'Changed rerank policy or cohort\')\n    if sha(out/\'inference_inputs.json\') != lock[\'inference_sha256\']:\n        raise ValueError(\'Changed inference inputs\')\n    return lock\n\n\ndef read_beams(out, lock):\n    out = Path(out)\n    test = out.parent\n    inputs = read(out/\'inference_inputs.json\')\n    words = read(test/\'vocabulary.json\')\n    by_id = {}\n    for rank in (0, 1):\n        path = test/f\'beams_rank{rank}.json\'\n        receipt = read(test/f\'worker{rank}.json\')\n        if (not receipt[\'complete\'] or receipt[\'images\'] != 2500\n                or receipt[\'checkpoint_sha256\'] != lock[\'checkpoint_sha256\']\n                or sha(path) != receipt[\'beams_sha256\']):\n            raise ValueError(\'Incomplete CSA beam replay\')\n        rows = read(path)\n        original = read(test/f\'captions_rank{rank}.json\')\n        if len(original) != 2500 or [r[\'coco_id\'] for r in rows] != [r[\'coco_id\'] for r in inputs[rank::2]]:\n            raise ValueError(\'Wrong TEST beam coverage\')\n        for row, top in zip(rows, original):\n            beams = row[\'candidates\']\n            if (len(beams) != 5 or row[\'coco_id\'] != top[\'coco_id\']\n                    or beams[0] != dict(ids=top[\'ids\'], caption=top[\'caption\'], logprob=top[\'score\'])):\n                raise ValueError(\'Top-1 does not match beam candidates\')\n            if any(c[\'caption\'] != text_from_ids(c[\'ids\'], words) for c in beams):\n                raise ValueError(\'Changed beam text\')\n            by_id[row[\'coco_id\']] = row\n    return [by_id[row[\'coco_id\']] for row in inputs]\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    prepare(read(parser.parse_args().config))\n',encoding="utf-8")
    (SOURCE/'run_csa_per_caption_rerank.py').write_text("import argparse, hashlib, io, json, os, time\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom csa_per_caption_rerank_data import checked_lock, read_beams, POLICIES\nfrom captioning.multiscore_reranking import rerank_multiscore\nfrom score_prompt_test_extra import java_path\n\ndef fixed_rank(row, clipscore, detected):\n    if len(row['candidates']) != 5 or len(clipscore) != 5 or any((not np.isfinite(x) or not 0 <= x <= 2.50001 for x in clipscore)):\n        raise ValueError('Invalid TEST CLIP scores/width')\n    scores = [c['logprob'] for c in row['candidates']]\n    if not all(np.isfinite(scores)) or scores != sorted(scores, reverse=True):\n        raise ValueError('Invalid beam logprobs')\n    enriched = [dict(c, clipscore=float(s)) for c, s in zip(row['candidates'], clipscore)]\n    chosen, _ = rerank_multiscore(dict(candidates=enriched), detected, 0.4, 0.3)\n    return int(chosen['original_rank'])\n\ndef prefetch(cfg):\n    from huggingface_hub import snapshot_download\n    out = Path(cfg['output'])\n    locked = checked_lock(out)['clip']\n    root = Path(snapshot_download(locked['id'], revision=locked['revision'], allow_patterns=list(locked['files_sha256']), cache_dir=cfg['model_cache']))\n    for n, h in locked['files_sha256'].items():\n        if sha(root / n) != h:\n            raise ValueError('CLIP assets differ from VALID164: ' + n)\n    save(out / 'clip_assets.json', dict(**locked, path=str(root)))\n    print('CLIP model/processor match frozen VALID164 bytes; no BLIP download.', flush=True)\n\ndef text_preflight(cfg):\n    from transformers import CLIPProcessor\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    rows = read_beams(out, lock)\n    model = read(out / 'clip_assets.json')\n    processor = CLIPProcessor.from_pretrained(model['path'], local_files_only=True, use_fast=False)\n    lengths = [len(v) for v in processor.tokenizer([c['caption'] for r in rows for c in r['candidates']], truncation=False)['input_ids']]\n    if len(lengths) != 25000 or max(lengths) > 77:\n        raise ValueError('Would truncate TEST candidates')\n    save(out / 'token_coverage.json', dict(images=5000, captions=25000, max_tokens=max(lengths), limit=77))\n\ndef worker(cfg):\n    import torch\n    import torch.nn.functional as F\n    from PIL import Image\n    from transformers import CLIPModel, CLIPProcessor\n    rank = int(os.environ['LOCAL_RANK'])\n    if int(os.environ['WORLD_SIZE']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError('Select T4x2')\n    torch.cuda.set_device(rank)\n    device = torch.device('cuda', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    asset = read(out / 'clip_assets.json')\n    for n, h in lock['clip']['files_sha256'].items():\n        if sha(Path(asset['path']) / n) != h:\n            raise ValueError('Changed CLIP model/processor')\n    base = CLIPModel.from_pretrained(asset['path'], local_files_only=True, use_safetensors=False, attn_implementation='eager').float().to(device).eval().requires_grad_(False)\n    processor = CLIPProcessor.from_pretrained(asset['path'], local_files_only=True, use_fast=False)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256']:\n        raise ValueError('Changed TEST inference inputs')\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    started = time.monotonic()\n    count = 0\n    path = out / f'scores_rank{rank}.jsonl'\n    with path.open('x', encoding='utf-8') as log, torch.inference_mode():\n        for row, beam in list(zip(inputs, beams))[rank::2]:\n            if time.monotonic() - started > cfg['phase_seconds']:\n                raise TimeoutError('TEST CLIP scoring cap; preserve partial scores')\n            raw = Path(row['image_path']).read_bytes()\n            if hashlib.sha256(raw).hexdigest() != row['image_sha256']:\n                raise ValueError('Changed raw TEST image')\n            with Image.open(io.BytesIO(raw)) as image:\n                image = image.convert('RGB')\n            pixels = processor(images=image, return_tensors='pt')['pixel_values'].to(device)\n            text = processor.tokenizer([c['caption'] for c in beam['candidates']], padding=True, truncation=False, return_tensors='pt').to(device)\n            image_features = F.normalize(base.get_image_features(pixel_values=pixels).float(), dim=-1)\n            text_features = F.normalize(base.get_text_features(**text).float(), dim=-1)\n            clipscore = (2.5 * (image_features * text_features).sum(-1).clamp_min(0)).cpu().tolist()\n            chosen = fixed_rank(beam, clipscore, row['detected'])\n            log.write(json.dumps(dict(coco_id=row['coco_id'], image_sha256=row['image_sha256'], clipscore=clipscore, selected_rank=chosen), allow_nan=False) + '\\n')\n            log.flush()\n            count += 1\n            if count % 100 == 0:\n                print('TEST CLIP GPU', rank, count, '/2500', flush=True)\n            if count == 25 and (time.monotonic() - started) / count * (2500 - count) > cfg['phase_seconds'] - (time.monotonic() - started):\n                raise TimeoutError('Measured TEST CLIP projection exceeds cap')\n    save(out / f'worker{rank}.json', dict(complete=True, images=count, scored_pairs=count * 5, scores_sha256=sha(path), seconds=time.monotonic() - started, torch_version=torch.__version__, training=False, clip_revision=lock['clip']['revision']))\n\ndef score(cfg):\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    java_path(Path(cfg['test_output']))\n\n    import subprocess, sys\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from saved_quality_data import chair_counts\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256'] or sha(out / 'test_references.json') != lock['references_sha256']:\n        raise ValueError('Changed locked inputs')\n    scored = {}\n    for rank in (0, 1):\n        w = read(out / f'worker{rank}.json')\n        path = out / f'scores_rank{rank}.jsonl'\n        if not w['complete'] or w['images'] != 2500 or w['scored_pairs'] != 12500 or (sha(path) != w['scores_sha256']):\n            raise ValueError('Incomplete TEST scoring worker')\n        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]\n        if [r['coco_id'] for r in rows] != [r['coco_id'] for r in inputs[rank::2]]:\n            raise ValueError('Wrong TEST score IDs')\n        scored.update({r['coco_id']: r for r in rows})\n    if len(scored) != 5000:\n        raise ValueError('Missing TEST scores')\n    chosen = []\n    for row, beam in zip(inputs, beams):\n        s = scored[row['coco_id']]\n        if s['image_sha256'] != row['image_sha256']:\n            raise ValueError('Wrong image-score identity')\n        k = fixed_rank(beam, s['clipscore'], row['detected'])\n        if k != s['selected_rank']:\n            raise ValueError('Fixed selection replay differs')\n        chosen.append(k)\n    save(out / 'selected_ranks.json', chosen)\n    refs = read(out / 'test_references.json')\n    ptb = PTBTokenizer()\n    ref = ptb.tokenize({r['coco_id']: [dict(caption=c) for c in r['references']] for r in refs})\n    chair = Path(cfg['chair'])\n    metrics = {}\n    flags = {}\n    if sha(chair / 'chair.pkl') != '73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':\n        raise ValueError('Changed CHAIR cache')\n    for policy in POLICIES:\n        selected = [dict(coco_id=r['coco_id'], caption=r['candidates'][0 if policy == 'decoder_rank0' else k]['caption']) for r, k in zip(beams, chosen)]\n        hyp = ptb.tokenize({r['coco_id']: [dict(caption=r['caption'])] for r in selected})\n        c, per = Cider().compute_score(ref, hyp)\n        b, _ = Bleu(4).compute_score(ref, hyp)\n        r, _ = Rouge().compute_score(ref, hyp)\n        ci = out / (policy + '_chair_input.json')\n        co = out / (policy + '_chair_output.json')\n        save(ci, [dict(image_id=r['coco_id'], caption=r['caption']) for r in selected])\n        subprocess.run([sys.executable, '-u', chair / 'chair.py', '--cap_file', ci, '--cache', chair / 'chair.pkl', '--save_path', co], cwd=chair, check=True, timeout=300)\n        ch = read(co)\n        m, counts = chair_counts(ch, selected)\n        meteor = Meteor()\n        try:\n            meteor_score, meteor_per = meteor.compute_score(ref, hyp)\n        finally:\n            del meteor\n        spice_score, spice_per = Spice().compute_score(ref, hyp)\n        if len(meteor_per) != 5000 or len(spice_per) != 5000 or not np.isfinite([meteor_score, spice_score]).all():\n            raise ValueError('Incomplete METEOR/SPICE')\n        metrics[policy] = dict(CIDEr=float(c), BLEU_1=float(b[0]), BLEU_4=float(b[3]), METEOR=float(meteor_score), SPICE=float(spice_score), ROUGE_L=float(r), **m, counts=counts, mean_words=sum((len(r['caption'].split()) for r in selected)) / 5000, short_le2=sum((len(r['caption'].split()) <= 2 for r in selected)))\n        save(out / (policy + '_captions.json'), selected)\n        save(out / (policy + '_cider_per_image.json'), [dict(coco_id=int(i), CIDEr=float(v)) for i, v in zip(ref, per)])\n        flags[policy] = {r['image_id']: bool(r['mscoco_hallucinated_words']) for r in ch['sentences']}\n    a, b = [flags[p] for p in POLICIES]\n    changes = dict(changed_captions=sum((k != 0 for k in chosen)), rank_histogram={str(k): chosen.count(k) for k in range(5)}, CHAIR_repairs=sum((a[i] and (not b[i]) for i in a)), CHAIR_new_flags=sum((not a[i] and b[i] for i in a)))\n    result = dict(status='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE', images=5000, test_used=False, training=False, checkpoint_sha256=lock['checkpoint_sha256'], metrics=metrics, changes=changes, selector=lock['selector'], tuned_on_test=False, oracle_used_for_selection=False, prior_project_test_exposure=True, selection='both prelocked policies reported; fixed VALID164 policy', NLL_note='same frozen captioner; reranking changes output only, no target-conditioned pass')\n    save(out / 'result.json', result)\n    print(json.dumps(result, indent=2), flush=True)\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('mode', choices=('prefetch', 'text_preflight', 'worker', 'score'))\n    parser.add_argument('config')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n",encoding='utf8')

    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():
        download(base+name,digest,CHAIR/name)
    run([sys.executable,'-u','-c',
         'import sys; from pathlib import Path; from score_prompt_test_extra import environment; environment(Path(sys.argv[1]))',OUTPUT],1200)
    rerank_config=SOURCE/'rerank_config.json'
    rerank_config.write_text(json.dumps(dict(output=str(rerank_out),test_output=str(OUTPUT),chair=str(CHAIR))))
    script=SOURCE/'run_csa_per_caption_rerank.py'
    run([sys.executable,'-u',script,'score',rerank_config],3600)
    result=json.loads((rerank_out/'result.json').read_text())
    if result['status']!='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE':raise ValueError('Incomplete rerank scoring')
    xe9 = {'CIDEr': 1.236969320694076, 'CHAIRs': 0.0346, 'CHAIRi': 0.023370070375780108, 'Recall': 0.44161225514816677}
    selected = result['metrics']['clip_object_fixed']
    if result['checkpoint_sha256'] != receipt['sha256'] or result['selector'] != {'decoder_weight': 0.3, 'clip_weight': 0.4, 'object_weight': 0.3, 'min_detection_confidence': 0.5, 'hallucination_penalty': 1.0, 'tie_break': 'earlier beam'}:
        raise ValueError('Checkpoint or selector differs from comparison protocol')
    scst1000 = {'CIDEr': 1.2468805966365206, 'CHAIRs': 0.0396, 'CHAIRi': 0.026118438065683994, 'Recall': 0.4493345052737318}
    final = {k:selected[k] for k in scst1000}
    comparison=dict(images=5000,split='val',xe9_fixed_rerank=xe9,
                    scst1000_fixed_rerank=scst1000,scst1500_fixed_rerank=final,
                    delta_vs_1000={k:final[k]-v for k,v in scst1000.items()},
                    delta_vs_xe9={k:final[k]-v for k,v in xe9.items()},
                    chair_s_cap=.042, meets_gate=final['CIDEr']>scst1000['CIDEr'] and final['CHAIRs']<=.042)
    (OUTPUT/'comparison.json').write_text(json.dumps(comparison,indent=2))
    print(json.dumps(comparison,indent=2),flush=True)
    complete=True
except BaseException:
    (OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_used=False,test_tuning=False,source_version=SOURCE_VERSION)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and not p.is_relative_to(OUTPUT/'assets') and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP OUTPUT:',OUTPUT,flush=True)
    print('SCST1500 CPU scoring complete.' if complete else 'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.',flush=True)
